# ComentaVenta IA — Análisis exploratorio del dataset

Explora el dataset sintético `data/raw/comentarios.csv` generado por el script `src.data.generate` a partir de las semillas en `data/raw/semillas.json` y expandido con GPT-4o-mini.

Objetivo: entender cómo se distribuye el dataset antes de entrenar los modelos.

- Distribución por clase (desbalance intencional)
- Longitud de los comentarios (muy cortos en comentarios de live)
- Palabras más frecuentes por clase
- Distribución por producto, estilo y origen
- Muestras aleatorias

In [ ]:
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Permite importar `src.*` cuando el notebook se corre desde /notebooks
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import LABELS, LABEL_DISPLAY

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_colwidth', 120)

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'raw' / 'comentarios.csv')
print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head()

In [ ]:
# Chequeos básicos de integridad
print('Valores nulos por columna:')
print(df.isna().sum())
print()
print(f'Textos duplicados exactos: {df["texto"].duplicated().sum()}')
print(f'IDs únicos: {df["id"].is_unique}')

## 1. Distribución por clase

La distribución es intencionalmente desbalanceada: imita un live real, donde la mayoría de los comentarios son saludos y conversación casual (`Otros`), y pocos son accionables (`Pedido`, `Objecion`). Por eso usamos **F1 macro** como métrica.

In [ ]:
conteo = df['clase'].value_counts().reindex(LABELS)
proporcion = (conteo / len(df) * 100).round(1)
resumen_clase = pd.DataFrame({'conteo': conteo, 'porcentaje': proporcion})
resumen_clase

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colores = sns.color_palette('Set2', n_colors=len(LABELS))
bars = ax.bar([LABEL_DISPLAY[c] for c in LABELS], conteo.values, color=colores)
ax.set_title('Distribución de comentarios por clase')
ax.set_ylabel('Cantidad de comentarios')
for b, v, p in zip(bars, conteo.values, proporcion.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 15, f'{v} ({p}%)',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2. Longitud de los comentarios

Los comentarios de live son típicamente muy cortos (una palabra o una frase). Esto afecta la elección del vectorizador y los modelos: TF-IDF con unigramas + bigramas captura bien lo esencial sin sobreajustar.

In [ ]:
df['n_caracteres'] = df['texto'].str.len()
df['n_palabras'] = df['texto'].str.split().str.len()
df[['n_caracteres', 'n_palabras']].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df, x='clase', y='n_palabras', order=LABELS, ax=axes[0], palette='Set2')
axes[0].set_title('Palabras por comentario, por clase')
axes[0].set_xlabel('')
axes[0].set_ylabel('# palabras')
axes[0].tick_params(axis='x', rotation=20)
sns.histplot(df['n_palabras'], bins=30, ax=axes[1], color='#6aaa64')
axes[1].set_title('Histograma de palabras por comentario')
axes[1].set_xlabel('# palabras')
plt.tight_layout()
plt.show()

## 3. Palabras más frecuentes por clase

Tokenización mínima (lowercase + split por no-alfabéticos). Es solo para la exploración; el pipeline final usa `TextPreprocessor` con normalización, stop words y stemming/lematización.

Lo que esperamos ver:
- **Pedido**: "yo", "mío", "quiero", "separas", colores, tallas
- **Pregunta**: "cuanto", "hay", "color", signos de interrogación
- **PagoEnvio**: "envían", "delivery", "Tigo Money", nombres de barrios
- **Objecion**: "caro", "barato", "original", "estafa", "mercado"
- **Otros**: "hola", "saludos", nombres de barrios, emojis

In [ ]:
# Lista mínima de stop words solo para esta exploración.
STOP_EDA = set('''
a al algo ante aqui as con como de del desde donde el en ese esta este estos
la lo las los le les mi mis muy no o para pero por que se si sin su sus
tu tus un una unos unas y ya es son fue era eran ser hay he ha has han
'''.split())

def tokens(texto: str) -> list[str]:
    return [t for t in re.findall(r"[a-záéíóúñü]+", texto.lower())
            if t not in STOP_EDA and len(t) > 1]

top_por_clase = {}
for clase in LABELS:
    counter = Counter()
    for texto in df[df['clase'] == clase]['texto']:
        counter.update(tokens(texto))
    top_por_clase[clase] = counter.most_common(15)

for clase, top in top_por_clase.items():
    print(f'\n{LABEL_DISPLAY[clase]}:')
    for palabra, c in top:
        print(f'  {palabra:20s} {c}')

In [ ]:
# Gráfico de barras horizontal, top 10 por clase
fig, axes = plt.subplots(1, 5, figsize=(20, 5), sharex=False)
for ax, clase, color in zip(axes, LABELS, sns.color_palette('Set2', len(LABELS))):
    top = top_por_clase[clase][:10]
    palabras = [p for p, _ in top][::-1]
    conteos = [c for _, c in top][::-1]
    ax.barh(palabras, conteos, color=color)
    ax.set_title(LABEL_DISPLAY[clase])
    ax.set_xlabel('')
plt.suptitle('Top 10 palabras por clase (sin stop words básicas)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Distribución por producto, estilo y origen

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
df['producto'].fillna('(sin producto)').replace('', '(sin producto)').value_counts().plot.bar(ax=axes[0], color='#4c9eeb')
axes[0].set_title('Comentarios por producto')
axes[0].tick_params(axis='x', rotation=30)
df['estilo'].value_counts().plot.bar(ax=axes[1], color='#c76b8a')
axes[1].set_title('Comentarios por estilo')
axes[1].tick_params(axis='x', rotation=30)
df['origen'].value_counts().plot.bar(ax=axes[2], color='#8c8c8c')
axes[2].set_title('Comentarios por origen')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Cruce clase x estilo (quién usa qué forma de escribir)
cross = pd.crosstab(df['clase'], df['estilo']).reindex(LABELS)
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(cross, annot=True, fmt='d', cmap='YlGnBu', ax=ax)
ax.set_title('Comentarios por clase y estilo')
ax.set_xlabel('estilo')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 5. Muestras aleatorias por clase

Pequeño check cualitativo: ¿las semillas y las variantes GPT se ven mezcladas de forma natural? ¿Hay algo claramente mal etiquetado?

In [ ]:
for clase in LABELS:
    print(f'\n=== {LABEL_DISPLAY[clase]} ===')
    muestra = df[df['clase'] == clase].sample(8, random_state=42)
    for _, r in muestra.iterrows():
        marca = '●' if r['origen'] == 'semilla' else ' '
        print(f'  {marca} [{r["estilo"]:16s}] {r["texto"]}')

## Decisiones derivadas de este análisis

1. **Métrica**: `f1_macro` (ya está en `params.yaml`). La proporción Otros ~45% hace que la accuracy sea engañosa.
2. **Stop words**: la lista base de spaCy saca palabras clave para esta tarea (`yo`, `no`, `mío`, `caro`). Las conservamos vía `nlp.keep_words`.
3. **N-gramas**: comentarios muy cortos → unigramas + bigramas (TF-IDF). Los bigramas capturan "muy caro", "es original", "mío 50".
4. **Modelos**: Regresión logística + Naive Bayes multinomial. Ambos son fuertes con TF-IDF en textos cortos y permiten explicar la clasificación con pesos / log-prob por palabra.
5. **Baseline**: predecir siempre `Otros` (la clase más frecuente). Cualquier modelo debe superar un F1 macro de ~0.12.